# Autocorrect Tool — 03 Evaluation

We evaluate the corrector on a held-out list of common English misspellings (each paired with its intended word). Accuracy = fraction of misspellings mapped to the right word. All numbers are produced by running the code.

In [1]:
import pandas as pd
import utils
freq=utils.load_frequencies(); sc=utils.SpellCorrector(freq)
pairs=pd.read_csv('data/test_misspellings.csv')
print('test pairs:',len(pairs))
pairs.head()

test pairs: 43


,wrong,correct
0,speling,spelling
1,korrect,correct
2,recieve,receive
3,seperate,separate
4,definately,definitely


## 1. Run the evaluation

We score only pairs whose target word exists in the reference corpus (a corrector can never produce a word it has never seen).

In [2]:
inv=[(r.wrong,r.correct) for r in pairs.itertuples() if r.correct in freq]
acc,res=utils.evaluate(sc,inv)
print(f'evaluable pairs: {len(inv)}/{len(pairs)}')
print(f'correction accuracy: {acc:.3f}')

evaluable pairs: 42/43
correction accuracy: 0.881


## 2. Per-item results

In [3]:
res

,wrong,expected,predicted,correct
0,speling,spelling,spelling,True
1,korrect,correct,correct,True
2,recieve,receive,receive,True
3,seperate,separate,separate,True
4,definately,definitely,definitely,True
5,goverment,government,government,True
6,neccessary,necessary,necessary,True
7,becuase,because,because,True
8,beleive,believe,believe,True
9,begining,beginning,beginning,True


## 3. Error analysis

The misses show the method's limits — a 2-edit cap, or a more common word outranking the intended one.

In [4]:
print('MISSES:')
print(res[~res['correct']][['wrong','expected','predicted']].to_string(index=False))

MISSES:
wrong expected predicted
 thru  through      thru
 wich    which      with
  cud    could       cut
 wich    which      with
 alot        a       lot


## 4. Summary & takeaways

- The Norvig corrector reaches **~88% accuracy** on common misspellings using nothing but edit distance + word frequency — no training, no neural network.
- **Frequency is the key prior**: among equally-close candidates, the more common word is almost always the intended one.
- **Failure modes**: errors needing >2 edits, or cases where a higher-frequency word legitimately outranks the target (e.g. a short misspelling close to a very common word).
- **Limits**: the corrector is context-free — it cannot use surrounding words to disambiguate (e.g. 'their' vs 'there'); that would need a language model, as in the Statistical Language Modeling project.